In [8]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
import torch.nn.functional as F

In [9]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8
EPOCHS = 100
LR = 2e-5

In [10]:
class MELDFusionDataset(Dataset):
    def __init__(self, label_encoder, csv_path, audio_path, video_path, text_path):
        self.audio_path = audio_path
        self.video_path = video_path
        self.text_path = text_path
        self.df = pd.read_csv(csv_path)
        self.label_encoder = label_encoder

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        dia_id, utt_id = row["Dialogue_ID"], row["Utterance_ID"]
        text_inputs_path = os.path.join(self.text_path, f"dia_{dia_id}_utt_{utt_id}.npy")
        audio_inputs_path = os.path.join(self.audio_path, f"dia_{dia_id}_utt_{utt_id}.npy")
        video_inputs_path = os.path.join(self.video_path, f"dia_{dia_id}_utt_{utt_id}.npy")
        
        label = self.label_encoder.transform([row["Emotion"]])[0]
        if os.path.exists(text_inputs_path):
            text_input = torch.from_numpy(np.load(text_inputs_path)).float()
        else:
            text_input = torch.zeros(768)
        
        if os.path.exists(audio_inputs_path):
            audio_input = torch.from_numpy(np.load(audio_inputs_path)).float()
        else:
            audio_input = torch.zeros(768)

        if os.path.exists(video_inputs_path):
            video_input = torch.from_numpy(np.load(video_inputs_path)).float()
        else:
            video_input = torch.zeros(2048)
        

        return {
            "text_input" : text_input,
            "video_input" : video_input,
            "audio_input" : audio_input,
            "labels" : torch.tensor(label, dtype = torch.long)
        }

In [ ]:
class FusionClassifier(nn.Module):
    def __init__(self, text_dim = 768, audio_dim = 768, video_dim = 2048, hidden_dim = 512, num_of_classes = 7):
        super().__init__()
        self.text_proj = nn.Linear(text_dim, hidden_dim)
        self.audio_proj = nn.Linear(audio_dim, hidden_dim)
        self.video_proj = nn.Linear(video_dim, hidden_dim)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim*3, num_of_classes)

    def forward(self, text_emb, audio_emb, video_emb):
        t = torch.relu(self.text_proj(text_emb))
        a = torch.relu(self.audio_proj(audio_emb))
        v = torch.relu(self.video_proj(video_emb))
        fused = torch.cat([v, a, t], dim = 1)
        fused = self.dropout(fused)
        return self.fc(fused)

In [12]:
def evaluate(model, data_loader, epoch):
    criterion = nn.CrossEntropyLoss()
    model.eval()
    y_true, y_pred = [], []
    total_loss = 0.0

    with torch.no_grad():
        for batch in tqdm(data_loader,desc=f"Epoch {epoch+1}/{EPOCHS} [Dev]"):
            # move batch to device
            batch = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in batch.items()}

            text_emb = batch["text_input"]
            audio_emb = batch["audio_input"]
            video_emb = batch["video_input"]

            logits = model(text_emb, audio_emb, video_emb)
            loss = criterion(logits, batch["labels"])

            total_loss += loss.item()
            preds = logits.argmax(dim=1).cpu().numpy()

            y_pred.extend(preds)
            y_true.extend(batch["labels"].cpu().numpy())

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="weighted")
    return total_loss / len(data_loader), acc, f1


In [13]:
train_csv = "C:/Users/hp333/Desktop/Multimodel_emotion_detection/data/MELD.Raw/train/train_sent_emo.csv"
dev_csv = "C:/Users/hp333/Desktop/Multimodel_emotion_detection/data/MELD.Raw/dev/dev_sent_emo.csv"
audio_fetures_path = "C:/Users/hp333/Desktop/Multimodel_emotion_detection/data/fetures/train/audio"
text_fetures_path = "C:/Users/hp333/Desktop/Multimodel_emotion_detection/data/fetures/train/text"
video_fetures_path = "C:/Users/hp333/Desktop/Multimodel_emotion_detection/data/fetures/train/video"
audio_fetures_path_dev = "C:/Users/hp333/Desktop/Multimodel_emotion_detection/data/fetures/dev/audio"
text_fetures_path_dev = "C:/Users/hp333/Desktop/Multimodel_emotion_detection/data/fetures/dev/text"
video_fetures_path_dev = "C:/Users/hp333/Desktop/Multimodel_emotion_detection/data/fetures/dev/video"

le = LabelEncoder()
train_df = pd.read_csv(train_csv)
le.fit(train_df["Emotion"])

train_set = MELDFusionDataset(le, train_csv, audio_fetures_path, video_fetures_path, text_fetures_path)
dev_set = MELDFusionDataset(le, dev_csv, audio_fetures_path_dev, video_fetures_path_dev, text_fetures_path_dev)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
dev_loader = DataLoader(dev_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [14]:
model = FusionClassifier().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()
SAVE_DIR = "C:/Users/hp333/Desktop/Multimodel_emotion_detection/Model/checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)
checkpoint_path = os.path.join(SAVE_DIR, "fusion_checkpoint.pt")

start_epoch = 0
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    best_dev_f1 = checkpoint["best_dev_f1"]
    print(f"✅ Resuming training from epoch {start_epoch}")

for epoch in range(start_epoch, EPOCHS):
    model.train()
    y_true, y_pred = [], []
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        optimizer.zero_grad()
        text_emb = batch["text_input"].to(DEVICE)
        audio_emb = batch["audio_input"].to(DEVICE)
        video_emb = batch["video_input"].to(DEVICE)
        logits = model(text_emb, audio_emb, video_emb)
        loss = criterion(logits, batch["labels"].to(DEVICE))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1).detach().cpu().numpy()
        y_pred.extend(preds)
        y_true.extend(batch["labels"].cpu().numpy())
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="weighted")
    dev_loss, dev_acc, dev_f1 = evaluate(model, dev_loader, epoch)

    checkpoint_path = os.path.join(SAVE_DIR, "fusion_checkpoint.pt")
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_dev_f1": dev_f1
    }, checkpoint_path)

    print("💾 Checkpoint saved (can resume training)")
    print(
        f"Epoch {epoch+1}: "
        f"Train Loss={total_loss/ len(train_loader):.4f}, Acc={acc:.4f}, F1={f1:.4f} | "
        f"Dev Loss={dev_loss:.4f}, Acc={dev_acc:.4f}, F1={dev_f1:.4f}"
    )

✅ Resuming training from epoch 50


Epoch 51/100 [Dev]: 100%|██████████| 139/139 [00:15<00:00,  9.02it/s]


💾 Checkpoint saved (can resume training)
Epoch 51: Train Loss=1.0397, Acc=0.6476, F1=0.6086 | Dev Loss=1.2366, Acc=0.5744, F1=0.5208


Epoch 52/100 [Dev]: 100%|██████████| 139/139 [00:17<00:00,  8.08it/s]


💾 Checkpoint saved (can resume training)
Epoch 52: Train Loss=1.0386, Acc=0.6447, F1=0.6057 | Dev Loss=1.2391, Acc=0.5771, F1=0.5231


...

💾 Checkpoint saved (can resume training)
Epoch 98: Train Loss=0.9177, Acc=0.6886, F1=0.6614 | Dev Loss=1.2294, Acc=0.5681, F1=0.5371


Epoch 99/100 [Dev]: 100%|██████████| 139/139 [00:01<00:00, 80.58it/s]


💾 Checkpoint saved (can resume training)
Epoch 99: Train Loss=0.9146, Acc=0.6887, F1=0.6619 | Dev Loss=1.2465, Acc=0.5771, F1=0.5306


Epoch 100/100 [Dev]: 100%|██████████| 139/139 [00:01<00:00, 80.64it/s]


💾 Checkpoint saved (can resume training)
Epoch 100: Train Loss=0.9134, Acc=0.6883, F1=0.6613 | Dev Loss=1.2501, Acc=0.5744, F1=0.5267
